# APÊNDICE A — CÓDIGO DO EXPERIMENTO

Versão consolidada do código correspondente ao procedimento experimental descrito no TCC.

Foram mantidos apenas os blocos necessários para carregar e preparar os corpora, definir os prompts e modelos, executar as inferências, tratar as respostas, calcular as métricas e gerar as tabelas e figuras utilizadas no trabalho. Blocos exploratórios ou posteriormente removidos do escopo final, como matrizes de confusão e análise qualitativa de erros, não são incluídos nesta versão.

## A.1 Bibliotecas, armazenamento e configurações

In [ ]:
!pip install -q openai

import os, re, glob, time, json, random, sys
from datetime import datetime
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openai, sklearn

from openai import OpenAI
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PASTA = "/content/drive/MyDrive/tcc_fake_news_resultados"
RESULTADOS = f"{PASTA}/resultados"
CONTROLE = f"{PASTA}/controle"
os.makedirs(RESULTADOS, exist_ok=True)
os.makedirs(CONTROLE, exist_ok=True)

TEMPERATURA = 0
MAX_TOKENS_SAIDA = 16
MAX_WORKERS = 4
CHECKPOINT = 25
MAX_TENTATIVAS = 5

## A.2 Carregamento dos corpora

### A.2.1 Fake.br

In [ ]:
if not os.path.exists("Fake.br-Corpus"):
    !git clone --quiet https://github.com/roneysco/Fake.br-Corpus.git

BASE_FBR = "Fake.br-Corpus/full_texts"

def carregar_fakebr():
    r = []
    for pasta, rotulo in [("fake", 1), ("true", 0)]:
        for arq in sorted(glob.glob(f"{BASE_FBR}/{pasta}/*.txt")):
            with open(arq, encoding="utf-8") as f:
                texto = f.read()
            r.append({
                "id": f"fbr_{rotulo}_{os.path.basename(arq)}",
                "par": os.path.basename(arq).replace(".txt", ""),
                "texto": texto,
                "rotulo": rotulo,
            })
    return pd.DataFrame(r)

fakebr_bruto = carregar_fakebr()
print(len(fakebr_bruto))
print(fakebr_bruto.rotulo.value_counts())

### A.2.2 FakeTrueBR

In [ ]:
if not os.path.exists("FakeTrue.Br"):
    !git clone --quiet https://github.com/jpchav98/FakeTrue.Br.git

CAMINHO_FTBR = "FakeTrue.Br/FakeTrueBr_corpus.csv"

def carregar_faketruebr(caminho):
    original = pd.read_csv(caminho, sep=None, engine="python")
    r = []
    for i, linha in original.iterrows():
        for coluna, rotulo in [("fake", 1), ("true", 0)]:
            valor = linha.get(coluna, None)
            texto = None if pd.isna(valor) else str(valor)
            r.append({
                "id": f"ftbr_{rotulo}_{i}",
                "par": str(i),
                "texto": texto,
                "rotulo": rotulo,
            })
    return pd.DataFrame(r)

faketruebr_bruto = carregar_faketruebr(CAMINHO_FTBR)
print(len(faketruebr_bruto))
print(faketruebr_bruto.rotulo.value_counts())

## A.3 Verificação de integridade e preparação dos dados

Os registros duplicados por conteúdo são apenas contabilizados e preservados. Não é realizado rebalanceamento ou remoção automática de duplicatas textuais.

In [ ]:
def diagnostico(df, nome):
    t = df["texto"].fillna("").astype(str)
    pares = df.groupby("par")["rotulo"].nunique()
    return {
        "corpus": nome,
        "registros": len(df),
        "classe_0": int((df.rotulo == 0).sum()),
        "classe_1": int((df.rotulo == 1).sum()),
        "nulos": int(df.texto.isna().sum()),
        "vazios": int(t.str.strip().eq("").sum()),
        "ids_duplicados": int(df.id.duplicated().sum()),
        "textos_duplicados": int(t[t.str.strip().ne("")].duplicated().sum()),
        "pares_incompletos": int((pares < 2).sum()),
        "media_chars": round(float(t.str.len().mean()), 1),
        "mediana_chars": round(float(t.str.len().median()), 1),
        "max_chars": int(t.str.len().max()),
    }

diag = pd.DataFrame([
    diagnostico(fakebr_bruto, "Fake.br"),
    diagnostico(faketruebr_bruto, "FakeTrue.Br"),
])
display(diag)
diag.to_csv(f"{CONTROLE}/diagnostico_inicial.csv", index=False, encoding="utf-8-sig")

def preparar(df, nome):
    texto = df.texto.fillna("").astype(str)
    invalido = df.texto.isna() | texto.str.strip().eq("")
    removidos = df[invalido].copy()

    if len(removidos):
        removidos.to_csv(
            f"{CONTROLE}/removidos_sem_texto_{nome}.csv",
            index=False, encoding="utf-8-sig"
        )

    out = df[~invalido].copy()
    out["texto"] = out.texto.astype(str).str.strip()

    if out.id.duplicated().any():
        raise ValueError(f"{nome}: IDs duplicados.")
    if not set(out.rotulo.unique()).issubset({0, 1}):
        raise ValueError(f"{nome}: rótulos inesperados.")

    out = out.sort_values("id").reset_index(drop=True)
    out["ordem"] = np.arange(len(out))
    return out

fakebr = preparar(fakebr_bruto, "fakebr")
faketruebr = preparar(faketruebr_bruto, "faketruebr")

CORPORA = {
    "fakebr_full": fakebr,
    "faketruebr": faketruebr
}

for nome, df in CORPORA.items():
    print(
        nome, len(df),
        "falsas=", int((df.rotulo == 1).sum()),
        "verdadeiras=", int((df.rotulo == 0).sum()),
        "max_chars=", int(df.texto.str.len().max())
    )

# Verificação adicional da convenção adotada:
# 0 = notícia verdadeira
# 1 = notícia falsa
def verificar_convencao_rotulos(df, prefixo):
    rotulo_id = pd.to_numeric(
        df["id"].astype(str).str.extract(
            rf"^{prefixo}_([01])_", expand=False
        ),
        errors="coerce",
    )

    comparaveis = rotulo_id.notna()

    divergencias = (
        df.loc[comparaveis, "rotulo"].astype(int).to_numpy()
        != rotulo_id.loc[comparaveis].astype(int).to_numpy()
    ).sum()

    if divergencias:
        raise ValueError(
            f"Foram encontradas {divergencias} divergências "
            "entre o rótulo e a classe codificada no ID."
        )

    print(
        f"{prefixo}: convenção 0=verdadeira e 1=falsa "
        "verificada sem divergências."
    )

verificar_convencao_rotulos(fakebr, "fbr")
verificar_convencao_rotulos(faketruebr, "ftbr")

## A.4 Definição das estratégias de prompt

Os quatro prompts são utilizados em regime zero-shot e mantêm a mesma tarefa binária: `0 = notícia verdadeira` e `1 = notícia falsa`.

In [ ]:
P1 = '''Você é um classificador de notícias em português brasileiro.

Classifique a notícia abaixo quanto à sua veracidade considerando apenas o texto fornecido.

Classes:
1 = notícia falsa
0 = notícia verdadeira

Responda somente com 0 ou 1. Não explique.

Notícia:
"""{texto}"""
'''

P2 = '''Você é um classificador de notícias em português brasileiro.

Classifique a notícia abaixo quanto à sua veracidade considerando apenas o conteúdo
e as características do texto fornecido.

Considere os elementos abaixo em conjunto, sem tratar nenhum deles isoladamente
como prova de falsidade ou veracidade:
- coerência interna das informações;
- linguagem sensacionalista, alarmista ou apelativa;
- afirmações extraordinárias sem sustentação identificável no próprio texto;
- atribuições vagas ou pouco específicas;
- clareza na identificação de pessoas, locais, datas e acontecimentos;
- estrutura e tom do texto.

Classes:
1 = notícia falsa
0 = notícia verdadeira

Responda somente com 0 ou 1. Não explique.

Notícia:
"""{texto}"""
'''

P3 = '''Você é um classificador de notícias em português brasileiro.

Classifique a notícia abaixo quanto à sua veracidade considerando apenas o conteúdo
e as características do texto fornecido.

Considere os elementos abaixo em conjunto, sem tratar nenhum deles isoladamente
como prova de falsidade ou veracidade:
- coerência interna das informações;
- linguagem sensacionalista, alarmista ou apelativa;
- afirmações extraordinárias sem sustentação identificável no próprio texto;
- atribuições vagas ou pouco específicas;
- clareza na identificação de pessoas, locais, datas e acontecimentos;
- estrutura e tom do texto.

Notícias falsas podem imitar características de reportagens legítimas. Nomes próprios,
cargos, datas, valores, citações, referências a fontes ou menções a veículos jornalísticos
não devem ser considerados isoladamente como evidência de veracidade.

Da mesma forma, linguagem emocional, posicionamento político ou estilo pouco formal
não devem ser considerados isoladamente como evidência de falsidade.

Avalie o conjunto das características disponíveis no texto.

Classes:
1 = notícia falsa
0 = notícia verdadeira

Responda somente com 0 ou 1. Não explique.

Notícia:
"""{texto}"""
'''

P4 = """<tarefa>
Você é um classificador de notícias em português brasileiro.
Classifique a notícia quanto à sua veracidade considerando apenas o conteúdo
e as características do texto fornecido.
</tarefa>

<criterios>
Considere em conjunto:
- coerência interna das informações;
- linguagem sensacionalista, alarmista ou apelativa;
- afirmações extraordinárias sem sustentação identificável no próprio texto;
- atribuições vagas ou pouco específicas;
- clareza na identificação de pessoas, locais, datas e acontecimentos;
- estrutura e tom do texto.
Nenhum elemento isolado é prova de falsidade ou veracidade.
</criterios>

<calibracao>
Notícias falsas podem imitar reportagens legítimas. Nomes próprios, cargos, datas,
valores, citações, referências a fontes ou veículos jornalísticos não são, isoladamente,
evidência de veracidade.
Linguagem emocional, posicionamento político ou estilo pouco formal também não são,
isoladamente, evidência de falsidade.
Avalie o conjunto das características disponíveis no texto.
</calibracao>

<classes>
1 = notícia falsa
0 = notícia verdadeira
</classes>

<saida>
Responda somente com 0 ou 1. Não explique.
</saida>

<noticia>
{texto}
</noticia>
"""

PROMPTS = {
    "p1_minimo": P1,
    "p2_criterios": P2,
    "p3_calibrado": P3,
    "p4_xml": P4,
}

## A.5 Configuração dos modelos e da API

O Qwen3-32B foi executado em modo *non-thinking* por meio do sufixo `/no_think`. A chave da API é solicitada de forma interativa e não é armazenada no notebook.

In [ ]:
API_KEY = getpass("Chave da DeepInfra: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepinfra.com/v1/openai",
)

MODELOS = {
    "llama8b": {
        "id": "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
        "sufixo": "",
    },
    "llama70b": {
        "id": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "sufixo": "",
    },
    "qwen32b": {
        "id": "Qwen/Qwen3-32B",
        "sufixo": "/no_think",
    },
    "qwen235b": {
        "id": "Qwen/Qwen3-235B-A22B-Instruct-2507",
        "sufixo": "",
    },
}

TOTAL_TEXTOS = sum(len(x) for x in CORPORA.values())
TOTAL_CHAMADAS = TOTAL_TEXTOS * len(MODELOS) * len(PROMPTS)

print("Textos:", TOTAL_TEXTOS)
print("Chamadas planejadas:", f"{TOTAL_CHAMADAS:,}".replace(",", "."))

### A.5.1 Registro da configuração experimental

In [ ]:
config = {
    "data": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "classe_positiva": "fake=1",
    "classe_negativa": "true=0",
    "n_por_corpus": {k: len(v) for k, v in CORPORA.items()},
    "temperature": TEMPERATURA,
    "max_tokens_saida": MAX_TOKENS_SAIDA,
    "max_workers": MAX_WORKERS,
    "modelos": MODELOS,
    "prompts": PROMPTS,
    "sem_busca_externa": True,
    "sem_rag": True,
    "sem_ferramentas": True,
    "software": {
        "python": sys.version,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "openai": openai.__version__,
        "sklearn": sklearn.__version__,
    },
}

with open(f"{CONTROLE}/config_experimento.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

## A.6 Chamada da API e tratamento das respostas

O parser considera interpretável a saída que apresenta `0` ou `1` ao final da resposta, admitindo espaços em branco e um ponto opcional após o rótulo. Respostas fora desse padrão permanecem registradas, mas recebem `predito = None`.

In [ ]:
def extrair_rotulo(resposta):
    if resposta is None:
        return None

    m = re.search(
        r"\b([01])\.?\s*$",
        str(resposta).strip()
    )

    return int(m.group(1)) if m else None


def estrito(resposta):
    return (
        resposta is not None
        and bool(
            re.fullmatch(
                r"\s*[01]\s*",
                str(resposta)
            )
        )
    )


def uso_da_resposta(resp):
    u = getattr(resp, "usage", None)

    if u is None:
        return None, None, None, None

    custo = getattr(u, "estimated_cost", None)

    if custo is None and hasattr(u, "model_dump"):
        custo = u.model_dump().get(
            "estimated_cost"
        )

    return (
        getattr(u, "prompt_tokens", None),
        getattr(u, "completion_tokens", None),
        getattr(u, "total_tokens", None),
        custo,
    )


def chamar(texto, cfg_modelo, tpl):

    conteudo = tpl.format(
        texto=str(texto)
    )

    if cfg_modelo["sufixo"]:
        conteudo += (
            "\n\n"
            + cfg_modelo["sufixo"]
        )

    inicio = time.perf_counter()

    ultimo_erro = None
    erro_fatal = False

    for tentativa in range(MAX_TENTATIVAS):

        try:

            resp = client.chat.completions.create(
                model=cfg_modelo["id"],
                messages=[
                    {
                        "role": "user",
                        "content": conteudo
                    }
                ],
                temperature=TEMPERATURA,
                max_tokens=MAX_TOKENS_SAIDA,
            )

            bruto = (
                resp.choices[0]
                .message
                .content
            )

            pt, ct, tt, custo = (
                uso_da_resposta(resp)
            )

            return {
                "resposta_bruta": bruto,

                "predito":
                    extrair_rotulo(bruto),

                "formato_estrito":
                    estrito(bruto),

                "erro_api": None,
                "erro_fatal": False,

                "latencia_s": round(
                    time.perf_counter()
                    - inicio,
                    4
                ),

                "request_id":
                    getattr(
                        resp,
                        "id",
                        None
                    ),

                # NOVO:
                "modelo_retornado":
                    getattr(
                        resp,
                        "model",
                        None
                    ),

                "prompt_tokens": pt,
                "completion_tokens": ct,
                "total_tokens": tt,

                "estimated_cost_usd":
                    custo,
            }

        except Exception as e:

            ultimo_erro = str(e)

            erro_lower = (
                ultimo_erro.lower()
            )

            # Problemas de crédito,
            # pagamento ou quota.
            if any(
                termo in erro_lower
                for termo in [
                    "credit",
                    "billing",
                    "payment",
                    "balance",
                    "insufficient",
                    "quota",
                ]
            ):

                erro_fatal = True
                break

            # Problemas temporários.
            temporario = any(
                termo in erro_lower
                for termo in [
                    "429",
                    "rate limit",
                    "rate_limit",
                    "too many requests",
                    "timeout",
                    "temporarily",
                    "overload",
                    "connection",
                ]
            )

            if (
                temporario
                and tentativa
                    < MAX_TENTATIVAS - 1
            ):

                espera = min(
                    30,
                    (2 ** tentativa)
                    + random.random()
                )

                time.sleep(espera)

                continue

            break

    return {
        "resposta_bruta": None,
        "predito": None,
        "formato_estrito": False,

        "erro_api": ultimo_erro,
        "erro_fatal": erro_fatal,

        "latencia_s": round(
            time.perf_counter()
            - inicio,
            4
        ),

        "request_id": None,
        "modelo_retornado": None,

        "prompt_tokens": None,
        "completion_tokens": None,
        "total_tokens": None,
        "estimated_cost_usd": None,
    }

## A.7 Execução, checkpoint e retomada

Cada combinação entre corpus, modelo e prompt é salva progressivamente. Erros temporários de API são tentados novamente e os resultados já persistidos não são repetidos em uma retomada.

A variável `EXECUTAR_EXPERIMENTO` permanece como trava de segurança. Quando ativada, a execução percorre **todos os corpora, todos os modelos e todos os prompts**, correspondendo ao desenho experimental completo.

In [ ]:
def caminhos(
    corpus,
    modelo,
    prompt
):

    base = (
        f"{RESULTADOS}/"
        f"{corpus}__"
        f"{modelo}__"
        f"{prompt}"
    )

    return (
        base + "__parcial.csv",
        base + "__final.csv",
        base + "__erros_api.csv",
    )


def ler_parcial(path):

    if not os.path.exists(path):
        return pd.DataFrame()

    d = pd.read_csv(
        path,
        on_bad_lines="skip"
    )

    if "id" in d.columns:
        d = d.drop_duplicates(
            "id",
            keep="last"
        )

    return d


def anexar(path, linhas):

    if not linhas:
        return

    existe = (
        os.path.exists(path)
        and os.path.getsize(path) > 0
    )

    pd.DataFrame(linhas).to_csv(
        path,
        mode="a",
        header=not existe,
        index=False,
        encoding="utf-8-sig",
    )


def processar_linha(
    corpus,
    modelo,
    prompt,
    linha,
    cfg,
    tpl
):

    r = chamar(
        linha.texto,
        cfg,
        tpl
    )

    return {
        "corpus": corpus,
        "modelo": modelo,
        "prompt": prompt,

        "id": linha.id,
        "par": linha.par,
        "ordem": int(linha.ordem),

        "rotulo_real":
            int(linha.rotulo),

        "n_caracteres":
            len(linha.texto),

        **r,
    }


def executar_combinacao(
    corpus,
    dados,
    modelo,
    cfg,
    prompt,
    tpl
):

    parcial, final, arquivo_erros = (
        caminhos(
            corpus,
            modelo,
            prompt
        )
    )

    # Se já existe arquivo final,
    # não repete a combinação.
    if os.path.exists(final):

        print(
            "[já concluído]",
            os.path.basename(final)
        )

        return

    # Carrega somente resultados
    # efetivamente persistidos.
    anterior = ler_parcial(
        parcial
    )

    if len(anterior):

        processados = set(
            anterior["id"].astype(str)
        )

    else:

        processados = set()

    # IDs que ainda precisam
    # ser classificados.
    pendentes = dados[
        ~dados["id"]
        .astype(str)
        .isin(processados)
    ].copy()

    print(
        f"\n[{corpus} | "
        f"{modelo} | "
        f"{prompt}] "
        f"salvos={len(processados)} "
        f"pendentes={len(pendentes)}"
    )

    if len(pendentes) == 0:

        consolidado = (
            anterior
            .sort_values("ordem")
            .reset_index(drop=True)
        )

        consolidado.to_csv(
            final,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "Combinação concluída:",
            final
        )

        return

    buffer_resultados = []
    buffer_erros = []

    concluidos_sessao = 0
    erros_sessao = 0

    abortar_execucao = False

    executor = ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    )

    futures = {}

    try:

        # Submete as notícias pendentes.
        for _, row in (
            pendentes.iterrows()
        ):

            future = executor.submit(
                processar_linha,
                corpus,
                modelo,
                prompt,
                row,
                cfg,
                tpl,
            )

            futures[future] = row.id

        for future in as_completed(
            futures
        ):

            try:

                resultado = (
                    future.result()
                )

            except Exception as e:

                # Falha inesperada no worker.
                id_ = futures[future]

                row = pendentes[
                    pendentes.id == id_
                ].iloc[0]

                resultado = {
                    "corpus": corpus,
                    "modelo": modelo,
                    "prompt": prompt,

                    "id": row.id,
                    "par": row.par,
                    "ordem":
                        int(row.ordem),

                    "rotulo_real":
                        int(row.rotulo),

                    "n_caracteres":
                        len(row.texto),

                    "resposta_bruta":
                        None,

                    "predito":
                        None,

                    "formato_estrito":
                        False,

                    "erro_api":
                        f"WORKER: {e}",

                    "erro_fatal":
                        False,

                    "latencia_s":
                        None,

                    "request_id":
                        None,

                    "modelo_retornado":
                        None,

                    "prompt_tokens":
                        None,

                    "completion_tokens":
                        None,

                    "total_tokens":
                        None,

                    "estimated_cost_usd":
                        None,
                }

            # -------------------------
            # ERRO DE API
            # -------------------------

            if (
                resultado["erro_api"]
                is not None
            ):

                erros_sessao += 1

                registro_erro = (
                    resultado.copy()
                )

                registro_erro[
                    "data_erro"
                ] = datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                )

                buffer_erros.append(
                    registro_erro
                )

                # Salva histórico de erros,
                # mas NÃO salva no parcial.
                if (
                    len(buffer_erros)
                    >= CHECKPOINT
                ):

                    anexar(
                        arquivo_erros,
                        buffer_erros
                    )

                    buffer_erros = []

                # Se for falta de crédito,
                # interrompe a combinação.
                if resultado.get(
                    "erro_fatal",
                    False
                ):

                    abortar_execucao = True

                    print(
                        "\nERRO FATAL "
                        "(crédito/pagamento/quota)."
                    )

                    print(
                        "Resultados válidos "
                        "já obtidos serão salvos."
                    )

                    raise RuntimeError(
    "Execução interrompida por erro fatal "
    "de crédito/pagamento/quota."
)

                continue

            # -------------------------
            # RESPOSTA RECEBIDA
            # -------------------------

            # Aqui entram tanto:
            # - respostas válidas 0/1;
            # - respostas inválidas do modelo.
            #
            # Como a API respondeu,
            # o experimento daquela notícia
            # foi efetivamente realizado.

            buffer_resultados.append(
                resultado
            )

            concluidos_sessao += 1

            if (
                len(buffer_resultados)
                >= CHECKPOINT
            ):

                anexar(
                    parcial,
                    buffer_resultados
                )

                buffer_resultados = []

                total_aprox = (
                    len(processados)
                    + concluidos_sessao
                )

                print(
                    f"progresso: "
                    f"{total_aprox}/"
                    f"{len(dados)} "
                    f"| erros API nesta sessão: "
                    f"{erros_sessao}"
                )

    except KeyboardInterrupt:

        abortar_execucao = True

        print(
            "\nInterrupção manual."
        )

        print(
            "Salvando resultados "
            "já concluídos..."
        )

        anexar(
            parcial,
            buffer_resultados
        )

        anexar(
            arquivo_erros,
            buffer_erros
        )


        print(
            "Parcial preservado no Drive."
        )

        raise

    finally:

        # Salva o que ficou abaixo
        # do tamanho do checkpoint.
        anexar(
            parcial,
            buffer_resultados
        )

        anexar(
            arquivo_erros,
            buffer_erros
        )

        try:

            executor.shutdown(
                wait=True,
                cancel_futures=abortar_execucao
            )

        except Exception:

            pass

    # Reabre o parcial depois
    # dos checkpoints.
    d = ler_parcial(
        parcial
    )

    print(
        f"Salvos: {len(d)}/"
        f"{len(dados)}"
    )

    if erros_sessao:

        print(
            f"Erros de API nesta sessão: "
            f"{erros_sessao}"
        )

        print(
            "Esses IDs NÃO foram "
            "considerados concluídos."
        )

        print(
            "Ao executar novamente, "
            "eles serão tentados outra vez."
        )

    # Somente cria FINAL quando
    # todos os IDs tiverem
    # recebido resposta da API.
    if len(d) == len(dados):

        d = (
            d
            .drop_duplicates(
                "id",
                keep="last"
            )
            .sort_values("ordem")
            .reset_index(drop=True)
        )

        d.to_csv(
            final,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "CONCLUÍDO:",
            final
        )

    else:

        print(
            "Combinação ainda incompleta."
        )

        print(
            "Execute novamente para "
            "processar os IDs restantes."
        )

# Trava de segurança para evitar novas chamadas acidentais à API.
EXECUTAR_EXPERIMENTO = False

if EXECUTAR_EXPERIMENTO:

    for corpus, dados in CORPORA.items():

        for modelo, cfg in MODELOS.items():

            for prompt, tpl in PROMPTS.items():

                executar_combinacao(
                    corpus,
                    dados,
                    modelo,
                    cfg,
                    prompt,
                    tpl,
                )

else:

    print("Execução bloqueada.")
    print(
        "Altere EXECUTAR_EXPERIMENTO = True "
        "somente quando desejar executar o experimento completo."
    )

## A.8 Auditoria das respostas e cálculo das métricas

As respostas sem rótulo interpretável são contabilizadas separadamente e não participam do cálculo das métricas. A notícia falsa (`1`) é considerada a classe positiva.

In [ ]:
auditoria = []
metricas = []
invalidas_todas = []

for corpus, dados in CORPORA.items():

    for modelo in MODELOS:

        for prompt in PROMPTS:

            parcial, final, arquivo_erros = caminhos(
                corpus,
                modelo,
                prompt,
            )

            path = final if os.path.exists(final) else parcial

            if not os.path.exists(path):
                continue

            d = (
                pd.read_csv(path, on_bad_lines="skip")
                .drop_duplicates("id", keep="last")
            )

            validas = d.dropna(subset=["predito"]).copy()
            invalidas = d[d["predito"].isna()].copy()

            auditoria.append({
                "corpus": corpus,
                "modelo": modelo,
                "prompt": prompt,
                "esperado": len(dados),
                "salvo": len(d),
                "faltando": len(dados) - len(d),
                "validas": len(validas),
                "invalidas": len(invalidas),
            })

            if len(invalidas):
                invalidas_todas.append(
                    invalidas.assign(
                        corpus=corpus,
                        modelo=modelo,
                        prompt=prompt,
                    )
                )

            # As métricas finais são calculadas apenas quando
            # a combinação foi completamente concluída.
            if not os.path.exists(final) or not len(validas):
                continue

            validas["predito"] = validas["predito"].astype(int)
            validas["rotulo_real"] = validas["rotulo_real"].astype(int)

            metricas.append({
                "corpus": corpus,
                "modelo": modelo,
                "prompt": prompt,
                "n_total": len(d),
                "n_validas": len(validas),
                "n_invalidas": len(invalidas),
                "acuracia": round(
                    accuracy_score(
                        validas["rotulo_real"],
                        validas["predito"],
                    ),
                    4,
                ),
                "precisao": round(
                    precision_score(
                        validas["rotulo_real"],
                        validas["predito"],
                        zero_division=0,
                    ),
                    4,
                ),
                "recall": round(
                    recall_score(
                        validas["rotulo_real"],
                        validas["predito"],
                        zero_division=0,
                    ),
                    4,
                ),
                "f1": round(
                    f1_score(
                        validas["rotulo_real"],
                        validas["predito"],
                        zero_division=0,
                    ),
                    4,
                ),
            })


auditoria_df = pd.DataFrame(auditoria)
metricas_df = pd.DataFrame(metricas)

if invalidas_todas:
    respostas_invalidas_df = pd.concat(
        invalidas_todas,
        ignore_index=True,
    )
else:
    respostas_invalidas_df = pd.DataFrame()

display(auditoria_df)
display(metricas_df)

if len(respostas_invalidas_df):
    display(
        respostas_invalidas_df[
            [
                "corpus",
                "modelo",
                "prompt",
                "id",
                "rotulo_real",
                "resposta_bruta",
            ]
        ]
    )

auditoria_df.to_csv(
    f"{CONTROLE}/auditoria.csv",
    index=False,
    encoding="utf-8-sig",
)

metricas_df.to_csv(
    f"{CONTROLE}/metricas.csv",
    index=False,
    encoding="utf-8-sig",
)

respostas_invalidas_df.to_csv(
    f"{CONTROLE}/respostas_invalidas.csv",
    index=False,
    encoding="utf-8-sig",
)

## A.9 Geração das tabelas de resultados

In [ ]:
# ============================================================
# TABELAS LIMPAS DE MÉTRICAS POR PROMPT E MODELO
# ============================================================

from IPython.display import display, Markdown

# Usa apenas as métricas principais da classificação
metricas_tabela = metricas_df[
    [
        "corpus",
        "modelo",
        "prompt",
        "acuracia",
        "precisao",
        "recall",
        "f1",
    ]
].copy()

# Ordem fixa dos modelos e prompts
ordem_modelos = [
    "llama8b",
    "llama70b",
    "qwen32b",
    "qwen235b",
]

ordem_prompts = [
    "p1_minimo",
    "p2_criterios",
    "p3_calibrado",
    "p4_xml",
]

# Nomes mais amigáveis para exibição
nomes_modelos = {
    "llama8b": "Llama 3.1 8B",
    "llama70b": "Llama 3.3 70B",
    "qwen32b": "Qwen3-32B",
    "qwen235b": "Qwen3-235B-A22B",
}

nomes_prompts = {
    "p1_minimo": "P1 — Instrução direta",
    "p2_criterios": "P2 — Critérios linguísticos e textuais",
    "p3_calibrado": "P3 — Critérios com calibração",
    "p4_xml": "P4 — Estruturação em XML",
}

nomes_corpus = {
    "fakebr_full": "Fake.br",
    "faketruebr": "FakeTrueBR",
}

for corpus in ["fakebr_full", "faketruebr"]:

    display(Markdown(f"# {nomes_corpus[corpus]}"))

    for prompt in ordem_prompts:

        tabela = metricas_tabela[
            (metricas_tabela["corpus"] == corpus)
            & (metricas_tabela["prompt"] == prompt)
        ].copy()

        tabela["modelo"] = pd.Categorical(
            tabela["modelo"],
            categories=ordem_modelos,
            ordered=True,
        )

        tabela = tabela.sort_values("modelo")

        tabela["modelo"] = tabela["modelo"].map(nomes_modelos)

        tabela = tabela[
            ["modelo", "acuracia", "precisao", "recall", "f1"]
        ].rename(
            columns={
                "modelo": "Modelo",
                "acuracia": "Acurácia",
                "precisao": "Precisão",
                "recall": "Recall",
                "f1": "F1-score",
            }
        )

        tabela = tabela.set_index("Modelo")

        display(Markdown(f"### {nomes_prompts[prompt]}"))

        display(
            tabela.style.format(
                {
                    "Acurácia": "{:.4f}",
                    "Precisão": "{:.4f}",
                    "Recall": "{:.4f}",
                    "F1-score": "{:.4f}",
                }
            )
        )

## A.10 Geração das figuras de desempenho por modelo e corpus

In [ ]:
# ============================================================
# GRÁFICOS DO CAPÍTULO 4.4
# Desempenho sob diferentes formulações de prompt
#
# Uma figura para cada combinação MODELO x CORPUS.
#
# Em cada figura:
#   - Acurácia
#   - Precisão
#   - Recall
#   - F1-score
#
# Cada grupo contém as barras P1, P2, P3 e P4.
#
# Total: 8 figuras
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Pasta para salvar as figuras
# ------------------------------------------------------------

PASTA_FIGURAS_PROMPTS = f"{RESULTADOS}/figuras_prompts_por_modelo"
os.makedirs(PASTA_FIGURAS_PROMPTS, exist_ok=True)


# ------------------------------------------------------------
# Ordem utilizada nos gráficos
# ------------------------------------------------------------

ordem_modelos = [
    "llama8b",
    "llama70b",
    "qwen32b",
    "qwen235b",
]

ordem_corpus = [
    "fakebr_full",
    "faketruebr",
]

ordem_prompts = [
    "p1_minimo",
    "p2_criterios",
    "p3_calibrado",
    "p4_xml",
]

ordem_metricas = [
    "acuracia",
    "precisao",
    "recall",
    "f1",
]


# ------------------------------------------------------------
# Nomes amigáveis para exibição
# ------------------------------------------------------------

nomes_modelos = {
    "llama8b": "Llama 3.1 8B",
    "llama70b": "Llama 3.3 70B",
    "qwen32b": "Qwen3-32B",
    "qwen235b": "Qwen3-235B-A22B",
}

nomes_corpus = {
    "fakebr_full": "Fake.br",
    "faketruebr": "FakeTrueBR",
}

nomes_prompts = {
    "p1_minimo": "P1",
    "p2_criterios": "P2",
    "p3_calibrado": "P3",
    "p4_xml": "P4",
}

nomes_metricas = {
    "acuracia": "Acurácia",
    "precisao": "Precisão",
    "recall": "Recall",
    "f1": "F1-score",
}


# ------------------------------------------------------------
# Padrões visuais dos prompts
#
# As cores são obtidas automaticamente do padrão do Matplotlib.
# Os hachurados ajudam caso a figura seja impressa em escala
# de cinza.
# ------------------------------------------------------------

cores_padrao = plt.rcParams["axes.prop_cycle"].by_key()["color"]

cores_prompts = {
    prompt: cores_padrao[i]
    for i, prompt in enumerate(ordem_prompts)
}

hachuras = {
    "p1_minimo": "",
    "p2_criterios": "//",
    "p3_calibrado": "..",
    "p4_xml": "xx",
}


# ------------------------------------------------------------
# Função principal
# ------------------------------------------------------------

def gerar_grafico_modelo_corpus(df, modelo, corpus):

    dados = df[
        (df["modelo"] == modelo)
        &
        (df["corpus"] == corpus)
    ].copy()

    if dados.empty:
        print(f"Sem dados para {modelo} / {corpus}")
        return

    x = np.arange(len(ordem_metricas))
    largura = 0.18

    deslocamentos = (
        np.arange(len(ordem_prompts))
        - (len(ordem_prompts) - 1) / 2
    ) * largura

    fig, ax = plt.subplots(figsize=(11, 6.8))

    for i, prompt in enumerate(ordem_prompts):

        valores = []

        for metrica in ordem_metricas:
            linha = dados[dados["prompt"] == prompt]

            if linha.empty:
                valores.append(np.nan)
            else:
                valores.append(float(linha[metrica].iloc[0]))

        posicoes = x + deslocamentos[i]

        barras = ax.bar(
            posicoes,
            valores,
            width=largura,
            label=nomes_prompts[prompt],
            color=cores_prompts[prompt],
            edgecolor="black",
            linewidth=0.7,
            hatch=hachuras[prompt],
        )

        # Rótulos com 4 casas decimais
        ax.bar_label(
            barras,
            labels=[
                f"{v:.4f}" if not np.isnan(v) else ""
                for v in valores
            ],
            padding=3,
            fontsize=8,
            rotation=90,
        )

    ax.set_title(
        f"Desempenho do {nomes_modelos[modelo]} sob diferentes estratégias de prompt — {nomes_corpus[corpus]}",
        fontsize=13,
        pad=28,
    )

    ax.set_ylabel("Valor da métrica", fontsize=11)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [nomes_metricas[m] for m in ordem_metricas],
        fontsize=11,
    )

    # Mais espaço em cima
    ax.set_ylim(0, 1.16)
    ax.set_yticks(np.arange(0, 1.01, 0.1))

    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.set_axisbelow(True)

    # Legenda mais acima, fora da área do gráfico
    ax.legend(
        title="Estratégia de prompt",
        ncol=4,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.08),
        frameon=False,
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Reserva espaço para título + legenda
    fig.tight_layout(rect=[0, 0, 1, 0.92])

    nome_base = f"prompts_{modelo}_{corpus}"

    caminho_png = f"{PASTA_FIGURAS_PROMPTS}/{nome_base}.png"
    caminho_pdf = f"{PASTA_FIGURAS_PROMPTS}/{nome_base}.pdf"

    fig.savefig(caminho_png, dpi=300, bbox_inches="tight")
    fig.savefig(caminho_pdf, bbox_inches="tight")

    plt.show()

    print(f"Salvo: {caminho_png}")


# ============================================================
# GERAR AS OITO FIGURAS
# ============================================================

for modelo in ordem_modelos:

    for corpus in ordem_corpus:

        gerar_grafico_modelo_corpus(
            metricas_df,
            modelo,
            corpus,
        )